# 1. Data Preparation and Azure ML Data Asset

Build, validate, fingerprint, and publish a RAFT dataset from local domain documents. The notebook is an orchestration layer; reusable logic lives in `lib/`.

## Learning objectives

- Generate synthetic question, retrieved-document, and grounded-answer examples from local files.
- Treat train, validation, and test splits as immutable experimental assets.
- Stop publication on schema defects, duplicates, or cross-split leakage.
- Publish a versioned Azure ML `uri_folder` data asset with a SHA-256 lineage fingerprint.

> Production rule: source data, prompts, split seed, and generated labels are part of model lineage. Never overwrite a published version.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import sys

from dotenv import load_dotenv
from IPython.display import display
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "lib").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "lib").exists():
    raise RuntimeError("Start this notebook from the repository or notebooks directory")
sys.path.insert(0, str(PROJECT_ROOT))

from lib.config import AzureMLConfig
from lib.data import dataset_fingerprint, publish_data_asset, validate_dataset

load_dotenv(PROJECT_ROOT / ".env")

True

## Configuration

Authentication uses `DefaultAzureCredential`. For local development, run `az login`; in Azure ML, assign a managed identity. Define `AZURE_SUBSCRIPTION_ID`, `AZURE_RESOURCE_GROUP`, and `AZUREML_WORKSPACE_NAME` in the environment. Azure OpenAI settings are only required when regenerating examples.

In [ ]:
DATASET_DIR = PROJECT_ROOT / "data" / "training_data_raft"
DATA_ASSET_NAME = "raft-instance-security"
DATA_ASSET_VERSION = datetime.now(timezone.utc).strftime("%Y%m%d.%H%M%S")
RUN_DATA_GENERATION = True # make this false to skip data generation and use existing data in DATASET_DIR
SKIP_PDF_EXTRACTION = False # make this true to skip PDF extraction and use existing extracted text in DATASET_DIR
NUM_QUESTIONS = 5
ORACLE_PROBABILITY = 0.8

assert 0 <= ORACLE_PROBABILITY <= 1
DATASET_DIR

WindowsPath('c:/code/raft-finetuning-slm/data')

## Optional local generation

The existing generator reads PDFs or page-level text chunks, uses Azure OpenAI to create grounded labels, and writes deterministic 80/10/10 splits. Keep this switch off when reviewing an already generated dataset: synthetic generation is billable and should be independently audited before publication.

In [ ]:
if RUN_DATA_GENERATION:
    from lib import pdf_to_chunks, raft_datagen

    pdf_path = PROJECT_ROOT / "data" / "instance-security-best-practice.pdf"
    images_dir = PROJECT_ROOT / "data" / "images"
    chunks_dir = PROJECT_ROOT / "data" / "chunks"

    if not SKIP_PDF_EXTRACTION:
        pdf_to_chunks.extract_chunks(
            pdf_path,
            images_dir=images_dir,
            chunks_dir=chunks_dir,
        )

    chunks = raft_datagen.load_chunks_from_dir(chunks_dir)

    if not chunks:
        raise RuntimeError(f"No chunks found in {chunks_dir}")

    raft_datagen.generate_dataset(
        chunks,
        num_questions=NUM_QUESTIONS,
        p=ORACLE_PROBABILITY,
    )

    raft_datagen.save_datasets(raft_datagen.ds.to_pandas(), str(DATASET_DIR))
else:
    print(f"Using existing local splits in {DATASET_DIR}")

2026-08-03 18:23:39 [INFO] lib.pdf_to_chunks: Converting 'instance-security-best-practice.pdf' → page images in 'c:\code\raft-finetuning-slm\data\images\instance-security-best-practice'
2026-08-03 18:23:39 [INFO] lib.pdf_to_chunks: Saved 45 page image(s) to 'c:\code\raft-finetuning-slm\data\images\instance-security-best-practice'
2026-08-03 18:23:39 [INFO] lib.pdf_to_chunks: Extracting text from 45 page(s) via GPT Vision ...
Pages: 100%|██████████| 45/45 [00:00<00:00, 19113.28it/s]
2026-08-03 18:23:39 [INFO] lib.pdf_to_chunks: Wrote 45 chunk file(s) to 'c:\code\raft-finetuning-slm\data\chunks\instance-security-best-practice'
2026-08-03 18:23:40 [INFO] lib.raft_datagen: Loaded 45 page file(s) from 'c:\code\raft-finetuning-slm\data\chunks'
2026-08-03 18:23:40 [INFO] lib.raft_datagen: 167 chunk(s) after splitting and filtering
2026-08-03 18:23:40 [INFO] lib.raft_datagen: Starting dataset generation for 167 chunks (num_questions=5, num_distract=3, p=0.80)
Processing chunks:   4%|▍         

: 

## Data quality gate

Validation checks required fields, non-empty learning signals, supported sample types, exact duplicate question/evidence pairs, and leakage across splits. The fingerprint changes if any split byte changes.

In [4]:
quality_summary = validate_dataset(DATASET_DIR)
display(pd.DataFrame(quality_summary).T)
print("Dataset fingerprint:", dataset_fingerprint(DATASET_DIR))

,rows,types,duplicates
train,23,"{'distractor': 3, 'oracle': 20}",0
validation,5,{'oracle': 5},0
test,5,"{'oracle': 4, 'distractor': 1}",0


Dataset fingerprint: 6bf5e18516b67216e1bf711a9b3223b61ee47dd829792588dfe86895262720a2


## Publish the versioned data asset

Azure ML uploads the folder to the workspace datastore (Azure Storage) and registers its metadata in the workspace asset catalog. The generated `manifest.json` captures row counts, class balance, creation time, and fingerprint.

In [5]:
config = AzureMLConfig.from_env()
ml_client = config.create_ml_client()

data_asset = publish_data_asset(
    ml_client=ml_client,
    dataset_dir=DATASET_DIR,
    name=DATA_ASSET_NAME,
    version=DATA_ASSET_VERSION,
    description="RAFT train/validation/test splits for instance-security grounded QA",
)

print(f"Published azureml:{data_asset.name}:{data_asset.version}")
print("Storage URI:", data_asset.path)

ValueError: Missing Azure ML configuration: subscription_id, resource_group, workspace_name